# Capstone Project: Personal Expense Tracker

This mini capstone pulls together everything covered in the Series module so
far. You have been logging every expense for over a year. The log lives in
`expenses.csv` (1,236 transactions) and, like all real logs, it is messy:

- Amounts are **strings** with a leading `₦`, typed by hand
- Category labels have **inconsistent casing and stray spaces**
- Some amounts were **never recorded** and come in blank
- The **date** is the natural row identifier

Your job is to turn this raw log into real answers using Series tools only.

**The project will only consist of concepts learned so far**:

- Reading a CSV column into a Series; `.values`, `.index`, `.name`, `.dtype`,
  `.size`
- `.astype()` type conversion
- Custom indices, `[]`, `.iloc`, `.loc`, `reset_index(drop=True)`
- Filtering with logical tests, `.isin()`, `~`, and masks
- `.sort_values()` and `.sort_index()`
- Numeric operations (operators, methods, `fill_value`)
- Text operations via the `.str` accessor
- Aggregations: `.sum`, `.mean`, `.median`, `.count`, `.min`, `.max`,
  `.std`, `.quantile`
- Categorical aggregation: `.unique`, `.nunique`, `.value_counts`
- Missing data: `.isna`, `.dropna`, `.fillna`


In [6]:
import numpy as np
import pandas as pd

## Loading the Data

Read the CSV. We pull two columns out as separate Series, each indexed by the
transaction date. Working with two aligned Series like this is a preview of
what a DataFrame does for us later.

The `date` column is set as the index, so a single transaction is addressable
by its date and the two Series stay aligned.

In [7]:
raw = pd.read_csv("https://raw.githubusercontent.com/Okwybobby/NDI-Python-Class/refs/heads/main/expenses.csv", index_col = "date")
raw.head()

,amount,category
date,,
2023-01-01,₦800.38,coffee
2023-01-02,₦1900.35,Groceries
2023-01-03,₦3700.70,dining
2023-01-03,₦800.70,Coffee
2023-01-03,₦2600.15,transport


### The amount column have Naira sign and there are missing values within the data which is represented as NaN

In [31]:
spend = pd.Series(raw["amount"].to_numpy(), index=raw.index, name="amount")
spend.head(10)

,amount
date,
2023-01-01,₦800.38
2023-01-02,₦1900.35
2023-01-03,₦3700.70
2023-01-03,₦800.70
2023-01-03,₦2600.15
2023-01-03,₦400.35
2023-01-04,₦6700.33
2023-01-04,₦2500.74
2023-01-04,₦5000.36


### To get the sum of the missing values in the amount column

In [14]:
spend.isna().sum()

np.int64(30)

In [19]:
missing = spend[spend.isna()]
missing

,amount
date,
2023-01-11,NaN
2023-02-03,NaN
2023-02-11,NaN
2023-03-07,NaN
2023-03-18,NaN
2023-03-19,NaN
2023-03-20,NaN
2023-04-07,NaN
2023-04-28,NaN


### The category column in its default state : unordered cases(mixture of lower and upper case) and leading & trailing spaces



In [20]:
category = pd.Series(raw["category"].to_numpy(), index=raw.index, name="category")
category.head(10)

,category
date,
2023-01-01,coffee
2023-01-02,Groceries
2023-01-03,dining
2023-01-03,Coffee
2023-01-03,transport
2023-01-03,Coffee
2023-01-04,groceries
2023-01-04,Dining
2023-01-04,Health


### Some key series properties

In [21]:
spend.size

1236

In [22]:
spend.dtype

dtype('O')

## **Challenge 1: Clean the Amounts**

The amounts are unusable for maths right now - they are Naira-prefixed
strings with some missing values.

1. Strip the `₦` from every amount and convert the Series to a float.
2. Confirm how many amounts were missing.
3. Fill the missing amounts with `0`, since a blank means you forgot to log the
   cost and we will treat it as no recorded spend.

Store the cleaned result back in `spend`.

### **Solution 1**

Strip the naira sign and cast to float. The missing values stay as NaN
through the conversion.

### Removing(strip()) the Naira Sign and changing the type from object(string) to float type and diplaying only the first 5 rows by using .head()

In [32]:
spend = spend.str.strip("₦").astype(float)
spend.head()

,amount
date,
2023-01-01,800.38
2023-01-02,1900.35
2023-01-03,3700.70
2023-01-03,800.70
2023-01-03,2600.15


## Count of the missing values within the amount column

In [19]:
spend.isna().sum()

np.int64(30)

##Getting the mean of the NaN value by dividing the total number of the NaN(30) to the total number of data(1206)

In [20]:
spend.isna().mean()

np.float64(0.024271844660194174)

In [25]:
##spend.isna().sum()

np.int64(30)

In [23]:
##spend.count()

np.int64(1206)

## The NaN(missing values) is replaced by zero(0) and check to confirm if there is still a missing value with the data

In [33]:
spend = spend.fillna(0)


In [29]:
spend.isna().sum()

np.int64(0)

# **Retured the clean data :**

*  Removed the Naira Sign
*   Removed the Spaces (trailing and leading)
*   filled the missing values(NaN) with zero(0)


In [34]:
spend

,amount
date,
2023-01-01,800.38
2023-01-02,1900.35
2023-01-03,3700.70
2023-01-03,800.70
2023-01-03,2600.15
...,...
2024-02-03,700.15
2024-02-03,2200.81
2024-02-04,8900.65


## **Challenge 2: Clean the Categories**

The category labels are inconsistent: mixed casing (`coffee`, `Coffee`) and
stray spaces (`"Groceries "`, `" groceries"`). Left as-is, the same category
would be counted as several different ones.

1. First, confirm how many *distinct raw labels* exist before cleaning.
2. Strip stray spaces from both ends of every label.
3. Convert every label to a single consistent case.
4. Confirm how many distinct categories you actually have, and list them.

## **Solution 2**

## Counting the number of unique values within our category

In [ ]:
category.nunique()

27

## Removing the leading and trailing spaces within the category column and lowering the case so that it will be uniform across

In [36]:
category = category.str.strip().str.lower()
category.head()

,category
date,
2023-01-01,coffee
2023-01-02,groceries
2023-01-03,dining
2023-01-03,coffee
2023-01-03,transport


## Total number of the unique value in category column

In [37]:
category.nunique()

8

## Diplay of the Unique values

In [38]:
category.unique()

array(['coffee', 'groceries', 'dining', 'transport', 'health',
       'utilities', 'shopping', 'rent'], dtype=object)

## **Challenge 3: The Headline Numbers**

Answer each question with a single aggregation on `spend`:

1. Total spent across the whole log.
2. Average spend per transaction.
3. The single largest transaction.
4. The typical (median) transaction, which ignores the big one-off costs.

### **Solution 3**

## Total amount of the whole transaction

In [23]:
spend.sum()

np.float64(4108045.66)

## Average spend per transaction ie. the mean

In [39]:
spend.mean()

np.float64(3323.6615372168285)

## The highest single transaction.



In [40]:
spend.max()

120000.0

## The Median transaction of the amount column

In [41]:
spend.median()

1900.32

## **Challenge 4: Where Did the Money Go?**

Get a feel for spending habits before totalling anything.

1. How often does each category appear in the log (raw counts)?
2. What share of your transactions does each category represent (as a
   proportion)?

### **Solution**
## Getting the frequency of each category across

In [42]:
category.value_counts()

,count
category,
coffee,358
groceries,276
transport,264
dining,148
shopping,86
utilities,55
health,44
rent,5


## Getting the percentage of each of the unique category

In [43]:
category.value_counts(normalize=True)

,proportion
category,
coffee,0.289644
groceries,0.223301
transport,0.213592
dining,0.119741
shopping,0.069579
utilities,0.044498
health,0.035599
rent,0.004045


## **Challenge 5: Spend Per Category**

Counts tell you how *often* you buy something, not how much it *costs*. Because
`spend` and `category` share the same index, a Boolean test on `category`
filters `spend` directly.

1. Total spent on groceries.
2. Total spent on coffee.
3. Total spent on everything that is **not** rent (use the tilde).

### **Solution 5**

## To get the total spent on groceries alone

In [44]:
spend.loc[category == "groceries"].sum()

np.float64(1294231.25)

## To get the total spent on coffee alone .

In [45]:
spend.loc[category == "coffee"].sum()

np.float64(214355.88)

## To get the total spent on everything apart from rent

In [46]:
spend.loc[~(category == "rent")].sum()

np.float64(3628045.66)

## **Challenge 6: The Discretionary Total**

You class coffee, transport, dining, and shopping as the spending you can
actually influence day to day. Rent, utilities, and health are effectively
fixed.

1. Use `.isin()` to total the spend across just those four categories.
2. What proportion of all spending did that discretionary spending make up?

### **Solution 6**

## Getting the total spent on only the discretionaries listed.

In [47]:
discretionary = ["coffee", "transport", "dining", "shopping"]
discretionary_total = spend.loc[category.isin(discretionary)].sum()
discretionary_total

np.float64(1637268.08)

## Dividing the total sum of the sum of the discretionaries to the total sum of all the transactions to get the proportion of the discretionary

In [48]:
discretionary_total / spend.sum()

np.float64(0.39855157792963775)

## **Challenge 7: Sorting to Find the Extremes**

1. Sort the spend Series chronologically by its date index.
2. Find the single largest transaction and the date it landed on. (Hint: sort
   by value, descending, and look at the top row.)
3. List the 5 largest transactions.

### **Solution 7**

## Sorting the data by the index which is the date

In [ ]:
spend.sort_index().head()

date
2023-01-01     800.38
2023-01-02    1900.35
2023-01-03    3700.70
2023-01-03     800.70
2023-01-03    2600.15
Name: amount, dtype: float64

Sorted by value, largest first. The top row's index label is the date of the
biggest hit.

## Sorting by value and in descending order to get the highest Transaction and the date it was made , before taking out the first value

In [50]:
spend.sort_values(ascending=False).head()

,amount
date,
2023-10-01,120000.00
2023-12-03,120000.00
2024-01-02,120000.00
2023-09-02,120000.00
2023-09-25,17300.16


## Displaying the single largest amount

In [49]:
spend.sort_values(ascending=False).iloc[0]

np.float64(120000.0)

## Displaying the 5 largest transactions still in descending order





In [ ]:
spend.sort_values(ascending=False).iloc[:5]

date
2024-01-02    120000.00
2023-09-02    120000.00
2023-12-03    120000.00
2023-10-01    120000.00
2023-09-25     17300.16
Name: amount, dtype: float64

## **Challenge 8: Filtering by Month**

The date index is text in the form `YYYY-MM-DD`, so the month sits in
positions 5:7. You can slice the index strings to filter by month.

1. Total spend in March (month "03") of any year.
2. How many transactions happened in the first quarter (months 01, 02, 03)?

### **Solution 8**

Build a month mask by slicing the index strings, then total March spend.

## Getting the total spent in month march and by first slicing with position(5:7)

In [ ]:
months = pd.Series(spend.index).str[5:7].to_numpy()
spend.loc[months == "03"].sum()

264344.17000000004

## The count of how many transactions were made within the first quarter

In [ ]:
q1_mask = pd.Series(spend.index).str[5:7].isin(["01", "02", "03"]).to_numpy()
spend.loc[q1_mask].count()

376

## **Challenge 9: Budget Check**

Your rule of thumb is that no everyday transaction should exceed `₦100`. Audit
the log against that. (Rent will obviously break it - that is expected.)

1. Build a mask for transactions over `₦100` and list the first few.
2. How many transactions broke the rule?
3. By how much, in total, did those transactions exceed the `₦100` limit?
   (Hint: subtract 100 from each over-limit amount, then sum.)

### **Solution 9**

## Display of transactions that the spend column is greater than 100 alone and limit the output  `

In [56]:
over_limit = spend.loc[spend > 100]
over_limit.head()

,amount
date,
2023-01-01,800.38
2023-01-02,1900.35
2023-01-03,3700.70
2023-01-03,800.70
2023-01-03,2600.15


# Count of how many transactions broke the rule

In [54]:
over_limit.count()

np.int64(1206)

# Total of all the transactions more than 100

In [ ]:
(over_limit - 100).sum()

3987445.66

## **Challenge 10: How Spread Out Is Your Spending?**

A single average hides how erratic spending is. Quantify the spread.

1. The standard deviation of transaction amounts.
2. The 25th, 50th, and 75th percentiles in one call.
3. The interquartile range (75th minus 25th), the band most spending falls in.

### Solution 10

## gettting the Standard deviation from the amount column

In [59]:
spend.std()

7238.577601289067

## Displaying the 20%,50% and 75% of the spend together

In [60]:
quartiles = spend.quantile([0.25, 0.5, 0.75])
quartiles

,amount
0.25,700.5875
0.50,1900.3200
0.75,4700.1750


## To get the band most spending fall on by substracting the last quartile(75%) by the first quartile(25%)

In [61]:
quartiles.loc[0.75] - quartiles.loc[0.25]

np.float64(3999.5875)

## **Challenge 11: Project Next Year**

Costs are rising. Build a rough forecast.

1. Apply a flat 8% increase to every transaction to model inflation.
2. On top of that, add a fixed `₦2` surcharge to each transaction.
3. Compare the projected total against the actual total, expressed as a
   percentage increase.

### **Solution 11**

Apply the 8% increase and the flat surcharge.

## Applying 8% ie.(100 * 0.08 = 1.08) to the spend and adding 2 Naira for each transaction

In [62]:
projected = spend * 1.08 + 2
projected.head()

,amount
date,
2023-01-01,866.4104
2023-01-02,2054.3780
2023-01-03,3998.7560
2023-01-03,866.7560
2023-01-03,2810.1620


## Comparing the Original total and the Projected total

In [63]:
spend.sum()

np.float64(4108045.66)

In [64]:
projected.sum()

np.float64(4439161.3127999995)

## TO get the percentage increase of the projected to the original total

In [ ]:
(projected.sum() - spend.sum()) / spend.sum()

0.08060174598935674

## Wrap-Up

Starting from a raw 1,236-row CSV, you cleaned messy real-world input (string
amounts and untidy labels), handled missing data, summarised the log with
aggregations, sliced spending by category using masks and `.isin()`, filtered
by month off the date index, audited against a budget rule, measured the
spread, and built a simple forecast - all with Series tools alone.

This is exactly the kind of work that becomes far smoother once these columns
live together in a single DataFrame, which is where the course heads next.